# Matched Filtering Cross Correlation

Matched filtering detects events missed by traditional phase picking methods. A template from an event is slid across continuous waveform data to find similar events by calculating a correlation coefficient at an event.

This example of matched filtering cross correlation uses the following steps:

1. Create an inventory of stations around the area of interest. Select stations with high frequency vertical channels that capture near source ground motion. 
2. Find known events for the area of interest within a range of dates and store them in a catalog.
3. The type of events can be mixed, adding a known volcanic event will improve the probablity of finding volcanic tectonic events. 
4. For each station, create picks based on the catalog of events.
5. Create 10 second templates that capture P and S waves.
6. Find new events by running the templates over continuous waveforms.


## Import packages

In [ ]:
from collections import defaultdict
import time

import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import (
    Catalog, Event, Pick, WaveformStreamID, Origin)
from obspy.geodetics import gps2dist_azimuth
from obspy import read
from obspy.core.inventory import (Inventory, Network, Station, Channel, 
                                   Site, Response)
from obspy.core.util import AttribDict
from parallel_download import download_miniseed_parallel

### Get Stations near Mount Saint Helens

The first step is to get an inventory of stations withing 0.25 km around the Mount Saint Helens summit. Volcanic earthquakes are typically weak motion, high frequency events, we're interest in high gain, vertical motion channels (HHZ).

In [ ]:
client = Client("https://service.earthscope.org")
inventory = client.get_stations(starttime=UTCDateTime(2014, 7, 17),
    endtime=UTCDateTime(2014, 7, 19),
    latitude=46.2, longitude=-122.19,
    maxradius=0.25, channel='*HZ', level='channel')

print(f"Number of stations: {len(inventory.get_contents()['stations'])}\n")

channel_count = 0
for network in inventory:
    for station in network:
        for channel in station:
            channel_count += 1

print(f"Total Channels Found: {channel_count}\n")

for channel in inventory.get_contents()['channels']:
    print(channel)

>**Explainer:**
>
>The obspy `get_stations` method is used to retrieve stations with a HHZ channel.

![stations](./images/stations.png)

We'll save the inventory which we can use as a list of miniseed data to download.

In [ ]:
inventory.write("./count_station.xml", format="stationxml", validate=True)

### Download Miniseed Data

#### Code Reuse

In the second notebook, [2_efficient_data_access](./2_efficient_data_access.ipynb), we wrote two functions to download miniseed data in parallel. The advantage of writing functions in notebooks is that they can be reused by moving them into standard Python script. Examine [parallel_download.py](./parallel_download.py) and you will see that it includes the package imports, the single file download function, and the parallel down load function. We can import the functions from the python file the same way other packages are imported.

```
from parallel_download import download_miniseed_parallel
```

We will use this function to download miniseed files for each station in the inventory.

In [ ]:


# convert inventory to dataframe
def inventory_to_dataframe(inventory):
    # convert inventory to dataframe used for downloading miniseed files
    data = []
    for network in inventory:
        for station in network:
            data.append({
                "network": network.code,
                "station": station.code,
                "latitude": station.latitude,
                "longitude": station.longitude,
                "elevation": station.elevation,
                "start_date": station.start_date,
                "end_date": station.end_date
            })
    return pd.DataFrame(data)

stations_df = inventory_to_dataframe(inventory)

# Start timer
start_time = time.time()

# download files
results = download_miniseed_parallel(
    stations_df.itertuples(),
    starttime='2014-07-17',
    endtime='2014-07-19',  # One day of data
    output_dir='./cross_correlation_data',
    max_workers=10  # Process 10 stations simultaneously
)

# Calculate and display elapsed time
elapsed_time = time.time() - start_time
print(f"\\nTotal execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")

>**Explainer:**
>
>This code converts seismic station information from am inventory structure into a flat pandas dataframe table where each row contains a station's details (network, location, dates). The dataframe format makes it easier to loop through stations and pass their information to the download function, which retrieves seismic data files for all stations in parallel to speed up the process.

#### Create A Local Database of Miniseed Files

For this example we will use [eqcorrscan](https://eqcorrscan.readthedocs.io/en/latest/) to perform matched filter cross correlation to find volcanic tectonic events. Eqcorrscan uses clients for data access, allowing it to request the data that it needs and take care of overlapping chunks of data to ensure that no data are missed. However we are using local data and the most efficient way to work with it is to create a local database using obsplus.

Next, create the miniseed database.

In [ ]:
from obsplus import WaveBank

bank = WaveBank("./cross_correlation_data").update_index()

bank.get_availability_df()

## Finding events

We can use obspy `get_events` to find earthquakes around Mount Saint Helens by searching for events within 0.25 km around the summit from 2014 to 2016 when the iMUSH data was collected.

In [ ]:
client = Client("USGS")
all_events_catalog = client.get_events(
    starttime=UTCDateTime(2014, 5, 1),
    endtime=UTCDateTime(2016, 12, 31),
    latitude=46.2, longitude=-122.19,
    maxradius=0.25)

print(all_events_catalog.count())

There are more than 1500 events in the iMUSH data. In this diagram of the obspy catalog, we can see that Event origins contains data we can use to find when earthquakes are ocurring frequently. Obspy catalogs are Python dictionaries and we can list the attribute names, or keys, to find the data we want.

![](./images/Event.png)

Obspy catalogs are Python dictionaries, we can find the keys for the items in the dictionary using `__dict__.keys()`.

In [ ]:
events = all_events_catalog[0].__dict__.keys()
for e in events:
    print(e)

> **Explainer:**
>
> Object attributes can be listed using a special dictionary, `__dict__` that stores the attributes of an object.

Using the catalog attribute, Event origins, we can examine the data by converting it to a pandas dataframe. The dataframe supports table operations for analyzing and displaying data. For example, we can sort the dataframe by time. Try sorting the dataframe by magnitude.

In [ ]:
df = pd.DataFrame([{
    'time': (event.origins[0]).time,
    'latitude': (event.origins[0]).latitude,
    'longitude': (event.origins[0]).longitude,
    'depth': (event.origins[0]).depth,
    'magnitude': (event.magnitudes[0]).mag if event.magnitudes else None
} for event in all_events_catalog])


sorted_df = df.sort_values(by=['magnitude'], ascending=[False])

display(sorted_df.head(10))

>**Explainer:**
>
>The event catalog is coverted to a pandas dataframe by accessing the catalog attributes. A dataframe canb display the catalog data in a table and supports operations such sorting, grouping, and calculating new values.
>
>`display` returns a formatted table, and `sorted_df.head(10)` shows only the first 10 events.

We can find dates with the greatest number of events by counting the number of events per day by grouping them and sorting by by descending values.

In [ ]:
from datetime import datetime

# convert UTCDateTime to Python datetime
df['dtm'] = df['time'].apply(lambda x: x.datetime)
# df['time'] = pd.to_datetime(df['time'])

# Extract just the date (without time) and add as a column
df['date'] = df['dtm'].dt.date

# After grouping and sorting
events_per_day = df.groupby('date').size().sort_values(ascending=False).head(10)

# Print all results
print(events_per_day)

>**Explainer:**
>
>Obspy uses UTCDateTime which is different from Python datetime. To work with dates in Python, UTCDateTime must be converted to a Python datetime object. In a dataframe, we can convert the entire `time` column with the `apply` method. The conversion uses a lambda, which is an inline function where x is the value to be converted by accessing the .datetime attribute of the ObsPy UTCDateTime object.
>
>To group events by date, a `date` column is created by extracting it from the `time` column.
>
>To count the events by day, we can use the `groupby` and `size` methods to count the number of occurrences by group, and display the top 5 days by the number of events per day.


### Create an event catalog.

In [ ]:
# get catalog of events

client = Client("USGS")
catalog = client.get_events(
    starttime=UTCDateTime(2014, 7, 17),
    endtime=UTCDateTime(2014, 7, 19),
    latitude=46.2, longitude=-122.19,
    maxradius=0.25)

print(catalog.count())

Events at volcanoes are difficult to locate, especially if they're small. If you don't have a known event, you'll never find them with template detection. [REDPy](https://code.usgs.gov/vsc/seis/tools/REDPy/) to find a known volcanic earthquake. We can take one event out of a family there to improve the probablity of finding like events.

The event triggered at 2014-07-25T16:57:57.08. There isn't a location, so we will assume it's near the summit at a depth of about 2 km b.s.l.

In [ ]:
event_solo = Event()
event_solo.origins = [Origin()]
event_solo.origins[0].time = UTCDateTime('2014-07-25T16:57:57.08') - 2  # assume about 2 seconds from origin to trigger time
event_solo.origins[0].latitude = 46.2
event_solo.origins[0].longitude = -122.19
event_solo.origins[0].depth = 2000

# Append to the catalog
catalog += event_solo

# print(catalog)
# print(catalog.__str__(print_all=True))

print(catalog.count())

>**Explainer**
>
>An event is created using the time of a known volcanic earthquake along with the location and depth. The event is added to the previous event catalog.

## Making templates

Unfortunately, pick information for these earthquakes are not available. We can make a very rough estimate of the P-wave arrival time at the stations of interest and use that as the start of our template window. In a large relative relocation study, you'd work with actual pick times and shorter windows around P- and S-, but for this example, this is a good starting point if you're looking to find more events.

### Add Picks

Loop over events and add picks for each channel based on distance of the station from source and a basic assumption of 5 km/s wave speed. Typically we would perform wave tracing throuhg a known velocity structure, however, this is a rough first pass. 

In [ ]:
# Loop over events and create picks for each channel based on distance of the station from source
# and a very simple assumption of 5 km/s wave speed. Could be more fancy by doing wave tracing
# through an actual velocity structure, but I'm just doing a rough first pass.

p_vel = 5000  # m/s 
for event in catalog:
    picks = []
    for channel in inventory.get_contents()['channels']:
        # XD (iMUSH deployment) broadbands have LHZ, BHZ, and HHZ channels; we only need HHZ
         if channel[:2] != 'XD' or (channel[:2] == 'XD' and channel[-3:] == 'HHZ'):
            distance = np.sqrt(gps2dist_azimuth(
                event.origins[0].latitude,
                event.origins[0].longitude,
                inventory.get_channel_metadata(channel)['latitude'],
                inventory.get_channel_metadata(channel)['longitude']
            )[0]**2 + (event.origins[0].depth + inventory.get_channel_metadata(channel)['elevation'])**2)
            picks += [
                Pick(
                    time=event.origins[0].time +  distance/p_vel,
                    waveform_id=WaveformStreamID(
                        network_code=channel.split('.')[0],
                        station_code=channel.split('.')[1],
                        channel_code=channel.split('.')[3],
                        location_code=channel.split('.')[2],
                    ),
                    phase_hint='P'
                )
            ]
    event.picks = picks
print(len(event.picks))

>**Explainer:**
>
>The code iterates through each event in a seismic catalog and generates synthetic P-wave arrival time picks for all available stations by calculating the travel time from each event's origin to each station. The 3D hypocentral distance between the event's origin location (latitude, longitude, depth) and the station's location (latitude, longitude, elevation) is calculated using the Pythagorean theorem on both the horizontal surface distance (calculated via gps2dist_azimuth) and the vertical depth difference.
>
>Using this distance and an assumed P-wave velocity (p_vel), it calculates the expected P-wave arrival time by adding the travel time (distance/velocity) to the event's origin time. It creates a Pick object with this synthetic arrival time and the channel's metadata (network, station, location, and channel codes extracted by splitting the dot-separated channel string) and marks it as a P-wave phase. It assigns the synthetic picks back to the event object, creating a theoretical pick catalog that assumes P-waves travel at a constant velocity.

### Creating Templates

[Eqcorrscan](https://eqcorrscan.readthedocs.io/en/latest/) is a Python package for the detection and analysis of repeating and near-repeating seismicity. Eqcorrscan provides a multi-parallel, matched-filter detection routine (template-matching) for detecting events.

### Create a Tribe

In matched filtering, a tribe is a collection of template waveforms that are used together to scan continuous seismic data for similar events. It is a container object (from the eqcorrscan library) that holds multiple Template objects. Each template represents a known seismic event's waveform pattern.

You may want to limit what picks you actually use for your templates. Any picks that you provide will be used for cutting waveforms. It maybe not be necessary to restrict what stations you have picks for, but it doesn’t do any harm to.

Below we select picks from the stations that we set earlier, and only P and S picks. We also limit to only one P and one S pick per station - you may not want to do that, but it can get messy if you have multiple picks of the same phase.

In [ ]:
from eqcorrscan.utils.catalog_utils import filter_picks

stations_df = bank.get_availability_df()
stations = list(stations_df['station'])

catalog = filter_picks(
    catalog,
    stations=stations,
    phase_hints=["P", "S"],
    enforce_single_pick="earliest")


We're constructing templates with a 10-second window to capture P and S with a bit of filtering. These can all be adjusted but should match the filtering and processing length you plan to use to scan through the continuous data. 

In [ ]:
import os
import obspy
from eqcorrscan import Tribe

# parameters for construct function: https://eqcorrscan.readthedocs.io/en/latest/submodules/core.match_filter.tribe.html#eqcorrscan.core.match_filter.tribe.Tribe.construct

"""
Method specific arguments:

from_client requires:
param str client_id:
string passable by obspy to generate Client, or any object with a get_waveforms method, including a Client instance.

param obspy.core.event.Catalog catalog:
Catalog of events to generate template for

param float data_pad:
Pad length for data-downloads in seconds

from_meta_file requires:
param str meta_file:
Path to obspy-readable event file, or an obspy Catalog

param obspy.core.stream.Stream st:
Stream containing waveform data for template. Note that this should be the same length of stream as you will use for the continuous detection, e.g. if you detect in day-long files, give this a day-long file!

param bool process:
Whether to process the data or not, defaults to True.
"""

tribe = Tribe().construct(
    method="from_client",
    client_id=bank,
    catalog=catalog,
    process=True,
    lowcut=1.0,
    highcut=10.0,
    samp_rate=50.0,
    filt_order=4,
    length=10.0,
    prepick=1,
    process_len=3600,
    all_horiz=False,
    parallel=True,
    min_snr=3.0,
)

print(len(tribe))

>**Explainer:**
>
>This code creates a Tribe of template waveforms by downloading seismic data from the Earthscope FDSN server for each event in the provided catalog. Each template is made by extracting:
>- 10-second waveform windows (starting 1 second before the pick time) from vertical channels only (since all_horiz=False),
>- resampling the data to 50 Hz, and
>- applying a 4th-order Butterworth bandpass filter between 1-10 Hz to isolate frequencies typical of volcanic-tectonic events.
>
> The method processes the waveforms in day-long segments (process_len=86400 seconds) and performs quality control by rejecting any templates with a signal-to-noise ratio below 3.0 (min_snr=3.0), ensuring only high-quality templates are included in the final tribe.
>
>By setting parallel=True, the construction process uses multiple CPU cores to simultaneously download and process waveforms for different events, significantly speeding up the tribe creation when working with large catalogs. Finally, the code prints the number of successfully created templates in the tribe, which may be less than the original catalog size if some events failed quality checks, had insufficient station coverage, or lacked available data on the Earthscope server.

We can plot the templates.

In [ ]:
print(tribe[0])
fig = tribe[0].st.plot(equal_scale=False, size=(800, 600))

We can examine the picks by converting the template events with picks to a dataframe.

In [ ]:
picks_data = []

for template in tribe:
    if template.event and template.event.picks:
        for pick in template.event.picks:
            pick_info = {
                'template_name': template.name,
                'station': pick.waveform_id.station_code,
                'channel': pick.waveform_id.channel_code,
                'phase': pick.phase_hint,
                'time': pick.time.datetime
            }
            picks_data.append(pick_info)

picks_df = pd.DataFrame(picks_data)
display(picks_df)

## Detecting Volcanic Earthquakes

To find other events, we run the templates through continuous data and see what wasn't located by the network. Recall that there were a large number for events from 7/25 to 8/3, we'll run the templates over this time period to find events attributable volcanic activity. This process will take some time because there's a *lot* going on under the hood.

In [ ]:
client = Client("https://service.earthscope.org")

party = tribe.client_detect(
    client=bank,
    starttime=UTCDateTime(2014, 7, 15),
    endtime=UTCDateTime(2014, 8, 4),
    threshold=10,
    threshold_type="MAD",
    trig_int=1.0,
    concurrent_processing=True,
)

>**Explainer:**
>
>The code performs matched filtering detection by scanning one day of continuous seismic data (from September 16-17, 2015) downloaded directly from the FDSN client. It searches for events similar to the templates stored in the tribe.
>
>The client_detect method automatically handles downloading the continuous waveforms from the server for the specified time period, cross-correlating each template against the continuous data, and identifying detections where the correlation coefficient exceeds a threshold of 10 times the Median Absolute Deviation (MAD), which is a robust statistical measure that adapts to the background noise level in the data.
>
>The `trig_int=1.0` parameter ensures that detections are separated by at least 1 second to prevent the same event from being detected multiple times. Eqcorrscan implements concurrent processing which enables parallel processing across multiple CPU cores to speed up the computationally intensive cross-correlation calculations.
>
>This function returns a Party object, which is a collection of detection Family objects. Each family contains all the detections found by a single template, along with metadata like detection times, correlation coefficients, and the associated template information, allowing you to see which template found which events and how strong the matches were.

As we've done before, we can extract the family from the party and convert the detections to a dataframe. 

In [ ]:
detections_data = []

for family in party:
    for detection in family:
        det_info = {
            'template_name': family.template.name,
            'detect_time': detection.detect_time.datetime,
            'detect_val': detection.detect_val,
            'threshold': detection.threshold,
            'detect_type': detection.typeofdet,
            'total_channels': detection.no_chans
        }
        detections_data.append(det_info)

detections_df = pd.DataFrame(detections_data).sort_values('detect_time')
display(detections_df)

Plot the party

In [ ]:
party.plot(plot_grouped=True)
party.plot(plot_grouped=False)

print(f'Number of templates: {len(tribe)}')
print(f'Number of additional detections: {len(party)-len(tribe)}')

We added some additional detections, though not a lot. Note that most of the templates are a mix of events from 2014_07_17 and 2014_07_18.  There are also templates that are matching each other, but we don't count those as additional detections since we're requiring that detections have to be at least 1 second apart from each other (trig_int).